# py-MARVEL droplet-based RNA-seq tutorial

This notebook rewrites the main MARVEL droplet tutorial from `MARVEL_Droplet.html` with the `marvel_py` 10x API. It uses the local tutorial data under `external_droplet_data/unpacked/Data`, which contains the processed iPSC and iPSC-derived cardiomyocyte droplet RNA-seq inputs from the original MARVEL tutorial.

The workflow follows the same practical order as the R tutorial:

1. inspect the required input files;
2. create a `Marvel10x` object;
3. annotate genes and splice junctions from the Cell Ranger GTF;
4. validate junctions, keep CDS/protein-coding genes, and align matrices;
5. define iPSC and day-10 cardiomyocyte cell groups;
6. summarize gene and splice-junction expression rates;
7. run differential splice-junction and gene analyses;
8. label differential results, assign iso-switch classes, and save reusable tables;
9. show optional pathway and candidate-gene analysis entry points.

Notes:

- The external droplet matrices are large. `marvel_py` writes `.csr.npz` caches next to Matrix Market files after the first load; this checkout already has those cache files.
- Differential SJ testing can be expensive on the full dataset. This tutorial defaults to a bounded tutorial subset and shows the switch for full-size analysis.
- Plot helpers in `marvel_py` store labeled tables and summaries; the notebook uses lightweight pandas/matplotlib displays for readability.


## 1. Setup

Run this notebook in the `ove` micromamba environment, where `marvel_py` is importable as a normal package. The bundled input and output directories are declared explicitly below.


In [ ]:
from pathlib import Path

import marvel_py as mp
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import sparse
from scipy.io import mmread

PROJECT_ROOT = Path('/data20T/dev/omicverse-eco/py-MARVEL')
DATA_ROOT = PROJECT_ROOT / 'external_droplet_data' / 'unpacked' / 'Data'
OUTPUT_DIR = PROJECT_ROOT / 'examples' / 'output' / 'droplet_data'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('project root:', PROJECT_ROOT)
print('data root:', DATA_ROOT)
print('output dir:', OUTPUT_DIR)


In [ ]:
mp.__file__


## 2. Input files

The droplet workflow uses three matrix families and their metadata:

- normalized gene expression from SingCellaR;
- raw gene counts from STARsolo;
- raw splice-junction counts from STARsolo;
- dimension-reduction coordinates for display;
- a Cell Ranger GTF for gene and splice-junction annotation.

Rows in the gene matrices are `gene_short_name`; rows in the SJ matrix are `coord.intron`; columns are `cell.id`.


### Download the tutorial data

The processed input files used by the original MARVEL droplet tutorial are available from:

- Main tutorial archive: <http://datashare.molbiol.ox.ac.uk/public/project/meadlab/wwen/MARVEL/Tutorial/Droplet/Data.zip>

Download and unpack it before running this notebook:

```bash
cd /data20T/dev/omicverse-eco/py-MARVEL
mkdir -p external_droplet_data/unpacked
wget -O external_droplet_data/Data.zip \
  http://datashare.molbiol.ox.ac.uk/public/project/meadlab/wwen/MARVEL/Tutorial/Droplet/Data.zip
unzip -o external_droplet_data/Data.zip -d external_droplet_data/unpacked
```


In [ ]:
paths = {
    'gene_norm_matrix': DATA_ROOT / 'Gene_SingCellaR' / 'matrix_normalised.mtx',
    'gene_norm_pheno': DATA_ROOT / 'Gene_SingCellaR' / 'phenoData.txt',
    'gene_norm_feature': DATA_ROOT / 'Gene_SingCellaR' / 'featureData.txt',
    'gene_count_matrix': DATA_ROOT / 'Gene_STARsolo' / 'matrix_counts.mtx',
    'gene_count_pheno': DATA_ROOT / 'Gene_STARsolo' / 'phenoData.txt',
    'gene_count_feature': DATA_ROOT / 'Gene_STARsolo' / 'featureData.txt',
    'sj_count_matrix': DATA_ROOT / 'SJ_STARsolo' / 'matrix_counts.mtx',
    'sj_count_pheno': DATA_ROOT / 'SJ_STARsolo' / 'phenoData.txt',
    'sj_count_feature': DATA_ROOT / 'SJ_STARsolo' / 'featureData.txt',
    'pca': DATA_ROOT / 'Gene_SingCellaR' / 'dim_red_coordinates_iPSC_CardioDay10.txt',
    'gtf': DATA_ROOT / 'GTF' / 'refdata-cellranger-GRCh38-3.0.0.gtf',
}

missing = [name for name, path in paths.items() if not path.exists()]
if missing:
    raise FileNotFoundError(f'Missing tutorial inputs: {missing}')

pd.DataFrame(
    [{'input': name, 'path': str(path.relative_to(PROJECT_ROOT)), 'size_mb': path.stat().st_size / 1024**2} for name, path in paths.items()]
).round({'size_mb': 2})


### Sample metadata

The normalized-gene metadata carries the tutorial cell annotations used to define groups. The raw STARsolo count metadata contain the same cell IDs used by the count matrices.


In [ ]:
gene_norm_pheno = pd.read_csv(paths['gene_norm_pheno'], sep='\t')
gene_count_pheno = pd.read_csv(paths['gene_count_pheno'], sep='\t')
sj_count_pheno = pd.read_csv(paths['sj_count_pheno'], sep='\t')

print(gene_norm_pheno.shape)
display(gene_norm_pheno.head())
display(gene_norm_pheno['cell.type'].value_counts().rename_axis('cell.type').reset_index(name='n_cells'))


### Feature metadata

Gene feature files require `gene_short_name`; the splice-junction feature file requires `coord.intron`.


In [ ]:
gene_norm_feature = pd.read_csv(paths['gene_norm_feature'], sep='\t')
gene_count_feature = pd.read_csv(paths['gene_count_feature'], sep='\t')
sj_count_feature = pd.read_csv(paths['sj_count_feature'], sep='\t')

print('gene features:', gene_norm_feature.shape)
print('SJ features:', sj_count_feature.shape)
display(gene_norm_feature.head())
display(sj_count_feature.head())


### Dimension-reduction coordinates and GTF

The R tutorial uses tSNE/UMAP-style coordinates generated before MARVEL. In `marvel_py`, these coordinates are carried in `marvel.pca` and can be joined to cell groups or expression values.


In [ ]:
pca = pd.read_csv(paths['pca'], sep='\t')
gtf = pd.read_csv(
    paths['gtf'],
    sep='\t',
    header=None,
    comment='#',
    dtype=str,
    names=[f'V{i}' for i in range(1, 10)],
)

print('pca:', pca.shape)
print('gtf:', gtf.shape)
display(pca.head())
display(gtf.head())


## 3. Create MARVEL object

`create_marvel_object_10x` takes the sparse matrices, metadata tables, coordinates, and GTF that were read above, then packages them into a `Marvel10x` object. This mirrors the R tutorial style: read files first, then pass the loaded objects into MARVEL.


In [ ]:
def read_sparse_matrix(path):
    cache_path = Path(f'{path}.csr.npz')
    if cache_path.exists():
        return sparse.load_npz(cache_path).tocsr()
    matrix = mmread(str(path))
    return matrix.tocsr() if sparse.issparse(matrix) else sparse.csr_matrix(matrix)


gene_norm_matrix = read_sparse_matrix(paths['gene_norm_matrix'])
gene_count_matrix = read_sparse_matrix(paths['gene_count_matrix'])
sj_count_matrix = read_sparse_matrix(paths['sj_count_matrix'])

marvel = mp.create_marvel_object_10x(
    gene_norm_matrix=gene_norm_matrix,
    gene_norm_pheno=gene_norm_pheno,
    gene_norm_feature=gene_norm_feature,
    gene_count_matrix=gene_count_matrix,
    gene_count_pheno=gene_count_pheno,
    gene_count_feature=gene_count_feature,
    sj_count_matrix=sj_count_matrix,
    sj_count_pheno=sj_count_pheno,
    sj_count_feature=sj_count_feature,
    pca=pca,
    gtf=gtf,
)


print('cells in normalized gene matrix:', marvel.gene_norm_matrix.matrix.shape[1])
print('genes in normalized gene matrix:', marvel.gene_norm_matrix.matrix.shape[0])
print('genes in raw count matrix:', marvel.gene_count_matrix.matrix.shape[0])
print('splice junctions in raw count matrix:', marvel.sj_count_matrix.matrix.shape[0])
print('sample metadata:', marvel.sample_metadata.shape)
print('pca coordinates:', marvel.pca.shape)


## 4. Pre-process data

The R tutorial annotates gene metadata, annotates junction metadata, validates junctions, keeps CDS genes, and then checks alignment across all matrices. The Python API exposes the same workflow with explicit 10x helper functions.


In [ ]:
marvel = mp.annotate_genes_10x(marvel)
marvel = mp.annotate_sj_10x(marvel)

print('annotated genes:', marvel.gene_metadata.shape)
print('annotated SJs:', marvel.sj_metadata.shape)
display(marvel.gene_metadata.head())
display(marvel.sj_metadata.head())


In [ ]:
# keep_novel_sj=False matches the strict known-start/known-end path used for the core tutorial.
marvel = mp.validate_sj_10x(marvel, keep_novel_sj=False)
marvel = mp.filter_genes_10x(marvel, gene_type='protein_coding')
marvel = mp.check_alignment_10x(marvel)

print('aligned normalized gene matrix:', marvel.gene_norm_matrix.matrix.shape)
print('aligned raw gene matrix:', marvel.gene_count_matrix.matrix.shape)
print('aligned SJ matrix:', marvel.sj_count_matrix.matrix.shape)
print('aligned sample metadata:', marvel.sample_metadata.shape)
print('aligned pca coordinates:', marvel.pca.shape)


## 5. Define cell groups

The original droplet tutorial compares iPSCs against iPSC-derived cardiomyocytes. This notebook uses iPSC versus day-10 cardiomyocytes, matching the bundled coordinate file name.


In [ ]:
GROUP_COLUMN = 'cell.type'
GROUP1 = 'iPSC'
GROUP2 = 'Cardio day 10'

cell_group_g1, cell_group_g2 = marvel.get_cell_groups(GROUP_COLUMN, GROUP1, GROUP2)
print(GROUP1, len(cell_group_g1))
print(GROUP2, len(cell_group_g2))

cell_group_list = {
    GROUP1: cell_group_g1,
    GROUP2: cell_group_g2,
}

display(pd.DataFrame([
    {'cell.group': group, 'n_cells': len(cells)}
    for group, cells in cell_group_list.items()
]))


## 6. Pre-flight visualization

`plot_values_pca_cell_group_10x` prepares a table joining coordinates with cell groups. The helper stores the result in `marvel.value_plots`; here we render a compact scatter plot with matplotlib.


In [ ]:
marvel = mp.plot_values_pca_cell_group_10x(
    marvel,
    cell_ids=cell_group_g1 + cell_group_g2,
    cell_group_column=GROUP_COLUMN,
    type='point',
)

pca_group = marvel.value_plots['pca_cell_group']['table']
fig, ax = plt.subplots(figsize=(6, 5))
for group, df_group in pca_group.groupby('group', sort=False):
    ax.scatter(df_group['x'], df_group['y'], s=4, alpha=0.45, label=group)
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.legend(markerscale=3, frameon=False)
ax.set_title('Tutorial cell groups')
plt.show()


## 7. Explore expression

The R tutorial first checks how many cells express each gene or splice junction in each group. The Python helpers produce the same table-style summaries.


In [ ]:
marvel = mp.plot_pct_expr_cells_genes_10x(
    marvel,
    cell_group_g1=cell_group_g1,
    cell_group_g2=cell_group_g2,
    min_pct_cells=1.0,
)

pct_gene = marvel.value_plots['pct_expr_genes']['table']
print(pct_gene.shape)
display(pct_gene.head())

fig, ax = plt.subplots(figsize=(6, 4))
for group, df_group in pct_gene.groupby('cell.group', sort=False):
    ax.hist(df_group['pct.cells.expr'], bins=40, alpha=0.45, label=group)
ax.set_xlabel('% cells expressing gene')
ax.set_ylabel('Number of genes')
ax.legend(frameon=False)
plt.show()


In [ ]:
marvel = mp.plot_pct_expr_cells_sj_10x(
    marvel,
    cell_group_g1=cell_group_g1,
    cell_group_g2=cell_group_g2,
    min_pct_cells_genes=10.0,
    min_pct_cells_sj=10.0,
    downsample=True,
    downsample_pct_sj=10.0,
    seed=1,
)

pct_sj = marvel.value_plots['pct_expr_sj']['table']
print(pct_sj.shape)
display(pct_sj.head())

fig, ax = plt.subplots(figsize=(6, 4))
for group, df_group in pct_sj.groupby('cell.group', sort=False):
    ax.hist(df_group['pct.cells.expr'], bins=40, alpha=0.45, label=group)
ax.set_xlabel('% cells expressing splice junction')
ax.set_ylabel('Number of splice junctions')
ax.legend(frameon=False)
plt.show()


## 8. Differential analysis

For differential splicing, MARVEL compares splice-junction usage between cell groups. The parameters mirror `external_droplet_data/droplet_ref.txt`; this notebook analyzes the most broadly expressed SJ subset for a practical tutorial run. Set `RUN_FULL_SJ_DE = True` and `SJ_DE_COORD_INTRONS = None` to run all validated junctions.


In [ ]:
RUN_FULL_SJ_DE = False
SJ_DE_N_INTRONS = 500
SJ_DE_ITERATIONS = 100

if RUN_FULL_SJ_DE:
    SJ_DE_COORD_INTRONS = None
else:
    SJ_DE_COORD_INTRONS = (
        pct_sj.groupby('coord.intron', as_index=False)['pct.cells.expr']
        .mean()
        .sort_values('pct.cells.expr', ascending=False)
        .head(SJ_DE_N_INTRONS)['coord.intron']
        .astype(str)
        .tolist()
    )

marvel = mp.compare_values_sj_10x(
    marvel,
    coord_introns=SJ_DE_COORD_INTRONS,
    cell_group_g1=cell_group_g1,
    cell_group_g2=cell_group_g2,
    min_pct_cells_genes=10.0,
    min_pct_cells_sj=10.0,
    min_gene_norm=1.0,
    seed=1,
    n_iterations=SJ_DE_ITERATIONS,
    downsample=True,
)

print(marvel.de_sj.shape)
display(marvel.de_sj.head())


In [ ]:
marvel = mp.compare_values_genes_10x(marvel, log2_transform=True, method='wilcox')

print(marvel.de_gene.shape)
display(marvel.de_gene.head())


## 9. Volcano-style labels

The Python plotting helpers label DE tables and store summaries. You can pass the stored tables to your preferred plotting library.


In [ ]:
PVAL = 0.05
DELTA_SJ = 5.0
MIN_GENE_NORM = 1.0
LOG2FC_GENE = 0.5

marvel = mp.plot_de_values_sj_10x(
    marvel,
    pval=PVAL,
    delta=DELTA_SJ,
    min_gene_norm=MIN_GENE_NORM,
)
marvel = mp.plot_de_values_genes_10x(
    marvel,
    pval=PVAL,
    log2fc=LOG2FC_GENE,
)

sj_labeled = marvel.de_plots['sj']['table']
gene_labeled = marvel.de_plots['genes']['table']

display(marvel.de_plots['sj']['summary'])
display(marvel.de_plots['genes']['summary'])


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9.5, 3.6), constrained_layout=True)
palette = {'n.s.': '0.75', 'up': '#D55E00', 'down': '#0072B2'}
draw_order = ['n.s.', 'down', 'up']
Y_CAP = 50

sj_plot = sj_labeled.copy()
sj_p_floor = 1.0 / (SJ_DE_ITERATIONS + 1.0)
sj_plot['minus_log10_p'] = -np.log10(sj_plot['pval'].clip(lower=sj_p_floor))
for sig in draw_order:
    df_sig = sj_plot[sj_plot['sig'] == sig]
    axes[0].scatter(
        df_sig['delta'],
        df_sig['minus_log10_p'],
        s=10,
        alpha=0.45 if sig == 'n.s.' else 0.8,
        color=palette[sig],
        label=sig,
        linewidths=0,
    )
axes[0].axvline(DELTA_SJ, color='0.45', lw=0.8, ls='--')
axes[0].axvline(-DELTA_SJ, color='0.45', lw=0.8, ls='--')
axes[0].axhline(-np.log10(PVAL), color='0.75', lw=0.8, ls=':')
axes[0].set_xlabel('Delta PSI (percentage points)')
axes[0].set_ylabel('-log10 p-value')
axes[0].set_title('Differential SJ')
axes[0].legend(frameon=False, loc='upper right')

gene_plot = gene_labeled.copy()
gene_p = gene_plot['pval.adj.gene.norm'].copy()
positive_min = gene_p[gene_p > 0].min()
gene_floor = positive_min if pd.notna(positive_min) else np.finfo(float).tiny
gene_plot['minus_log10_padj'] = (-np.log10(gene_p.clip(lower=gene_floor))).clip(upper=Y_CAP)
for sig in draw_order:
    df_sig = gene_plot[gene_plot['sig'] == sig]
    axes[1].scatter(
        df_sig['log2fc.gene.norm'],
        df_sig['minus_log10_padj'],
        s=10,
        alpha=0.45 if sig == 'n.s.' else 0.8,
        color=palette[sig],
        label=sig,
        linewidths=0,
    )
axes[1].axvline(LOG2FC_GENE, color='0.45', lw=0.8, ls='--')
axes[1].axvline(-LOG2FC_GENE, color='0.45', lw=0.8, ls='--')
axes[1].axhline(-np.log10(PVAL), color='0.75', lw=0.8, ls=':')
axes[1].set_xlabel('log2 fold-change')
axes[1].set_ylabel(f'-log10 adjusted p-value, capped at {Y_CAP}')
axes[1].set_title('Differential genes')
axes[1].legend(frameon=False, loc='upper right')

for ax in axes:
    ax.spines[['top', 'right']].set_visible(False)

plt.show()


## 10. Gene-splicing dynamics

`iso_switch_10x` classifies significant SJ changes relative to gene-level changes as coordinated, opposing, iso-switch, or complex. This mirrors the integrated gene-splicing dynamics section of the R tutorial.


In [ ]:
marvel = mp.iso_switch_10x(
    marvel,
    pval_sj=PVAL,
    delta_sj=DELTA_SJ,
    min_gene_norm=MIN_GENE_NORM,
    pval_adj_gene=PVAL,
    log2fc_gene=LOG2FC_GENE,
)

iso_table = marvel.iso_switch['table']
iso_summary = marvel.iso_switch['summary']
display(iso_summary)
display(iso_table.head())

summary_label_col = 'cor.complete' if 'cor.complete' in iso_summary.columns else 'sj.gene.cor'
fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(iso_summary[summary_label_col], iso_summary['freq'])
ax.set_ylabel('Number of genes')
ax.set_xlabel('Gene-splicing class')
ax.tick_params(axis='x', rotation=30)
plt.show()


## 11. Gene ontology analysis

GO enrichment from the original R tutorial used R annotation packages. Those R-backed pathway helpers are intentionally not part of this pure-Python tutorial, so this section is a placeholder for users who want to export DE tables to their preferred enrichment workflow.


In [ ]:
pathway_input = marvel.iso_switch['table'].copy() if marvel.iso_switch else pd.DataFrame()
print('Pathway input rows available for export:', len(pathway_input))


## 12. Candidate gene analysis

The R tutorial ends with ad hoc candidate-gene analysis. The Python API provides table builders for single-cell expression, pseudobulk expression, pseudobulk PSI, PCA overlays, and candidate-gene gene/SJ relationship summaries.


In [ ]:
# Pick a candidate from the significant iso-switch table when possible.
# Fallbacks are constrained to genes with a DE SJ first, so every downstream cell is runnable.
genes_with_sj = marvel.sj_metadata['gene_short_name.start'].astype(str).drop_duplicates().tolist()
genes_with_sj_set = set(genes_with_sj)
de_sj_table = marvel.de_sj.copy() if marvel.de_sj is not None else pd.DataFrame()
genes_with_de_sj = set(de_sj_table.get('gene_short_name', pd.Series(dtype=str)).dropna().astype(str))
preferred_genes = []
if not iso_table.empty:
    preferred_genes.extend(iso_table['gene_short_name'].dropna().astype(str).tolist())
preferred_genes.append('TNNT2')
if not de_sj_table.empty:
    preferred_genes.extend(de_sj_table['gene_short_name'].dropna().astype(str).head(20).tolist())
preferred_genes.extend(genes_with_sj[:10])

candidate_gene_pool = genes_with_de_sj or genes_with_sj_set
CANDIDATE_GENE = next(gene for gene in preferred_genes if gene in candidate_gene_pool)
candidate_sj_rows = de_sj_table.loc[de_sj_table.get('gene_short_name', pd.Series(dtype=str)).astype(str) == CANDIDATE_GENE].copy()
if not candidate_sj_rows.empty:
    candidate_sj_rows['abs_delta'] = candidate_sj_rows['delta'].abs()
    CANDIDATE_SJ = str(candidate_sj_rows.sort_values('abs_delta', ascending=False)['coord.intron'].iloc[0])
else:
    CANDIDATE_SJ = str(
        marvel.sj_metadata.loc[
            marvel.sj_metadata['gene_short_name.start'].astype(str) == CANDIDATE_GENE,
            'coord.intron',
        ].iloc[0]
    )

print('candidate gene:', CANDIDATE_GENE)
print('candidate SJ:', CANDIDATE_SJ)


In [ ]:
marvel = mp.plot_values_gene_single_cell_10x(
    marvel,
    cell_group_list=cell_group_list,
    gene_short_name=CANDIDATE_GENE,
    log2_transform=True,
)
marvel = mp.plot_values_gene_pseudobulk_10x(
    marvel,
    cell_group_list=cell_group_list,
    gene_short_name=CANDIDATE_GENE,
    log2_transform=True,
)
marvel = mp.plot_values_psi_pseudobulk_10x(
    marvel,
    cell_group_list=cell_group_list,
    coord_intron=CANDIDATE_SJ,
)
marvel = mp.plot_values_pca_gene_10x(
    marvel,
    gene_short_name=CANDIDATE_GENE,
    cell_ids=cell_group_g1 + cell_group_g2,
    log2_transform=True,
    type='point',
)
marvel = mp.plot_values_pca_psi_10x(
    marvel,
    coord_intron=CANDIDATE_SJ,
    cell_ids=cell_group_g1 + cell_group_g2,
    min_gene_count=3,
    log2_transform=False,
    type='point',
)

display(marvel.value_plots['gene_single_cell']['table'].head())
display(marvel.value_plots['gene_pseudobulk']['table'])
display(marvel.value_plots['psi_pseudobulk']['table'])


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

gene_pca = marvel.value_plots['pca_gene']['table']
sc = axes[0].scatter(gene_pca['x'], gene_pca['y'], c=gene_pca['gene_value'], s=5, cmap='viridis')
axes[0].set_title(f'{CANDIDATE_GENE} expression')
axes[0].set_xlabel('x')
axes[0].set_ylabel('y')
plt.colorbar(sc, ax=axes[0], fraction=0.046)

psi_pca = marvel.value_plots['pca_psi']['table']
sc = axes[1].scatter(psi_pca['x'], psi_pca['y'], c=psi_pca['psi'], s=5, cmap='magma')
axes[1].set_title('Candidate SJ PSI')
axes[1].set_xlabel('x')
axes[1].set_ylabel('y')
plt.colorbar(sc, ax=axes[1], fraction=0.046)

plt.tight_layout()
plt.show()


In [ ]:
marvel = mp.adhoc_gene_tabulate_expression_gene_10x(
    marvel,
    cell_group_list=cell_group_list,
    gene_short_name=CANDIDATE_GENE,
    log2_transform=True,
    min_pct_cells=10.0,
)
marvel = mp.adhoc_gene_tabulate_expression_psi_10x(marvel, min_pct_cells=10.0)
marvel = mp.adhoc_gene_de_gene_10x(marvel)
marvel = mp.adhoc_gene_de_psi_10x(marvel)
marvel = mp.adhoc_gene_plot_de_values_10x(
    marvel,
    coord_intron=CANDIDATE_SJ,
    log2fc_gene=LOG2FC_GENE,
    delta_sj=DELTA_SJ,
)
marvel = mp.adhoc_gene_plot_sj_position_10x(
    marvel,
    coord_intron=CANDIDATE_SJ,
    coord_intron_ext=25,
    show_protein_coding_only=True,
)

display(marvel.adhoc_gene['expression']['gene']['table'])
display(marvel.adhoc_gene['expression']['psi']['table'].head())
display(marvel.adhoc_gene['de']['volcano']['table'])
display(marvel.adhoc_gene['sj_position']['metadata'].head())
display(marvel.adhoc_gene['sj_position']['exonfile'].head())


## 13. Save tutorial outputs

This cell writes the main reusable tables generated by the notebook. It intentionally saves tables rather than pickling the full object, so downstream notebooks can reload compact TSV/JSON outputs.


In [ ]:
summary = {
    'group_column': GROUP_COLUMN,
    'group1': GROUP1,
    'group2': GROUP2,
    'n_cells_group1': len(cell_group_g1),
    'n_cells_group2': len(cell_group_g2),
    'n_genes_aligned': int(marvel.gene_norm_matrix.matrix.shape[0]),
    'n_sj_aligned': int(marvel.sj_count_matrix.matrix.shape[0]),
    'sj_de_full': RUN_FULL_SJ_DE,
    'sj_de_n_introns': None if SJ_DE_COORD_INTRONS is None else len(SJ_DE_COORD_INTRONS),
    'sj_de_iterations': SJ_DE_ITERATIONS,
    'candidate_gene': CANDIDATE_GENE,
    'candidate_sj': CANDIDATE_SJ,
}

marvel.save_outputs(OUTPUT_DIR, summary=summary)

extra_tables = {
    'de_sj_labeled.tsv': sj_labeled,
    'de_gene_labeled.tsv': gene_labeled,
    'iso_switch.tsv': iso_table,
    'iso_switch_summary.tsv': iso_summary,
    'candidate_gene_single_cell.tsv': marvel.value_plots['gene_single_cell']['table'],
    'candidate_gene_pseudobulk.tsv': marvel.value_plots['gene_pseudobulk']['table'],
    'candidate_psi_pseudobulk.tsv': marvel.value_plots['psi_pseudobulk']['table'],
}

for filename, table in extra_tables.items():
    table.to_csv(OUTPUT_DIR / filename, sep='\t', index=False)

for path in sorted(OUTPUT_DIR.glob('*')):
    print(path.relative_to(PROJECT_ROOT))


## 14. Where this differs from the R tutorial

This notebook intentionally focuses on the droplet workflow currently implemented and tested in `marvel_py`:

- The tutorial data path is local: `external_droplet_data/unpacked/Data`.
- Full differential SJ analysis is controlled by `RUN_FULL_SJ_DE`; the default uses a top-expressed SJ subset so the notebook is practical to run interactively.
- R-backed GO enrichment helpers are not used; DE and iso-switch tables can be exported for external enrichment tools.
- Python plotting helpers store labeled data and summaries; the notebook renders compact matplotlib views from those tables.
- The R tutorial's saved `MARVEL.RData` object is not used; all Python objects are created directly from the flat input files.
